In [2]:
import json
from datasets import load_dataset

In [ ]:
from huggingface_hub import login
login(token="huggin_face_key")

In [6]:
datasets = load_dataset('EdwardSJ151/portuguese_hate_speech_lighteval_fewshot', split="train[:13%]")

In [7]:
print(datasets)

Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 509
})


In [8]:
datasets = datasets.remove_columns(['idx'])

In [9]:
datasets = datasets.train_test_split(test_size=0.2)

In [10]:
print(datasets)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 407
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 102
    })
})


In [ ]:
def removeN(example):
  example['sentence'] = example['sentence'].replace("\n", " ")
  return example

datasets = datasets.map(removeN)

In [15]:
# label 0 -> No Hate Speech
# label 1 -> Hate Speech

def labelChange(example):
  example['label_text'] = 'No Hate Speech' if example['label'] == 0 else 'Hate Speech'
  return example

In [ ]:
datasets = datasets.map(labelChange)

In [17]:
datasets= datasets.remove_columns(['label'])

In [ ]:
print(datasets)

In [19]:
# Construindo o objeto para OpenAI

def dataset_to_jsonl(dataset, file_name):
  with open(file_name, 'w', encoding='utf-8') as file:
    for example in dataset:
      json_obj = {"messages": [
          {"role": "system", "content": "Seu trabalho é classificar os comentários do usuário em Hate Speech ou No Hate Speech."},
          {"role": "user", "content": example['sentence']},
          {"role": "assistant", "content": example['label_text']}
      ]}
      file.write(json.dumps(json_obj, ensure_ascii=False) + '\n')

In [20]:
dataset_to_jsonl(datasets['train'], 'train.jsonl')

In [21]:
dataset_to_jsonl(datasets['test'], 'validation.jsonl')

In [23]:
from openai import OpenAI
import os

In [ ]:
os.environ['OPENAI_API_KEY'] = "Openai_API_Key"

In [25]:
client = OpenAI()

In [26]:
client.files.create(
    file= open("train.jsonl", 'rb'),
    purpose= "fine-tune"
)

FileObject(id='file-JjyTB6mPMyByiSSecm2ybC', bytes=132121, created_at=1768250296, filename='train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [27]:
client.files.create(
    file= open("validation.jsonl", 'rb'),
    purpose= "fine-tune"
)

FileObject(id='file-QvByS7GRKUnzW4SX4TE5vS', bytes=33442, created_at=1768250313, filename='validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [28]:
client.fine_tuning.jobs.create(
    training_file= 'file-JjyTB6mPMyByiSSecm2ybC',
    validation_file= 'file-QvByS7GRKUnzW4SX4TE5vS',
    model= 'gpt-4.1-nano-2025-04-14'
)

FineTuningJob(id='ftjob-tYXdjEZB5aoQDnsnVFeHYQgC', created_at=1768251029, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-7BEzTGbv1UAu0bpQ4kzRuaMu', result_files=[], seed=559237818, status='validating_files', trained_tokens=None, training_file='file-JjyTB6mPMyByiSSecm2ybC', validation_file='file-QvByS7GRKUnzW4SX4TE5vS', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None)